# Entity, institution and country flow

**Ordering entity ? ordering entity country ? ordering institution ? ordering institution country ? receiving institution ? receiving institution country ? receiving entity ? receiving entity country.**

This sketch uses the original 23 sample transaction counts (total: **246**) with invented country and receiving-institution assignments. Each country field describes the participant immediately before it. Country nodes aggregate records in that stage; they are attributes of participants, not additional processing hops. Institutions and entities can have different countries.

Ribbons are coloured by receiving entity country. Hover a ribbon for its full sample route. Shared country nodes merge flows, so the overall Sankey shape alone does not establish a unique end-to-end path. Nodes remain freely draggable. The wide canvas keeps eight columns readable.

In [27]:
import plotly.graph_objects as go
from collections import defaultdict
from pathlib import Path

In [28]:
# Synthetic participant-country assignments; these are example records.
ordering_entity_countries = {
    "Ordering Entity A": "Germany",
    "Ordering Entity B": "France",
    "Ordering Entity C": "Netherlands",
    "Ordering Entity D": "United Kingdom",
    "Ordering Entity E": "Switzerland",
    "Ordering Entity F": "Germany",
}
ordering_institution_countries = {
    "Ordering Institution 1": "Germany",
    "Ordering Institution 2": "Germany",
    "Ordering Institution 3": "France",
    "Ordering Institution 4": "Netherlands",
    "Ordering Institution 5": "United Kingdom",
    "Ordering Institution 6": "Switzerland",
}
receiving_institution_countries = {
    "Receiving Institution 1": "United States",
    "Receiving Institution 2": "United Kingdom",
    "Receiving Institution 3": "Germany",
    "Receiving Institution 4": "Singapore",
    "Receiving Institution 5": "Switzerland",
    "Receiving Institution 6": "Netherlands",
    "Receiving Institution 7": "France",
}
receiving_entity_countries = {
    "Receiving Entity 1": "United States",
    "Receiving Entity 2": "United Kingdom",
    "Receiving Entity 3": "Singapore",
    "Receiving Entity 4": "Switzerland",
    "Receiving Entity 5": "Germany",
    "Receiving Entity 6": "Netherlands",
    "Receiving Entity 7": "France",
}

# Ordering entity, ordering institution, receiving institution, receiving entity,
# synthetic transaction count. The original 23 sample values are preserved.
sample_routes = [
    ("Ordering Entity A", "Ordering Institution 1", "Receiving Institution 1", "Receiving Entity 1", 42),
    ("Ordering Entity A", "Ordering Institution 1", "Receiving Institution 2", "Receiving Entity 2", 18),
    ("Ordering Entity A", "Ordering Institution 2", "Receiving Institution 1", "Receiving Entity 1", 22),
    ("Ordering Entity A", "Ordering Institution 2", "Receiving Institution 4", "Receiving Entity 3", 6),
    ("Ordering Entity A", "Ordering Institution 2", "Receiving Institution 5", "Receiving Entity 4", 5),
    ("Ordering Entity B", "Ordering Institution 3", "Receiving Institution 2", "Receiving Entity 2", 24),
    ("Ordering Entity B", "Ordering Institution 3", "Receiving Institution 1", "Receiving Entity 1", 20),
    ("Ordering Entity B", "Ordering Institution 3", "Receiving Institution 3", "Receiving Entity 5", 7),
    ("Ordering Entity B", "Ordering Institution 4", "Receiving Institution 5", "Receiving Entity 4", 8),
    ("Ordering Entity B", "Ordering Institution 4", "Receiving Institution 2", "Receiving Entity 2", 4),
    ("Ordering Entity C", "Ordering Institution 3", "Receiving Institution 1", "Receiving Entity 1", 14),
    ("Ordering Entity C", "Ordering Institution 3", "Receiving Institution 2", "Receiving Entity 2", 6),
    ("Ordering Entity C", "Ordering Institution 4", "Receiving Institution 2", "Receiving Entity 2", 10),
    ("Ordering Entity C", "Ordering Institution 4", "Receiving Institution 6", "Receiving Entity 6", 4),
    ("Ordering Entity D", "Ordering Institution 5", "Receiving Institution 5", "Receiving Entity 4", 9),
    ("Ordering Entity D", "Ordering Institution 5", "Receiving Institution 1", "Receiving Entity 1", 11),
    ("Ordering Entity D", "Ordering Institution 5", "Receiving Institution 4", "Receiving Entity 3", 3),
    ("Ordering Entity E", "Ordering Institution 6", "Receiving Institution 4", "Receiving Entity 3", 10),
    ("Ordering Entity E", "Ordering Institution 6", "Receiving Institution 2", "Receiving Entity 2", 6),
    ("Ordering Entity E", "Ordering Institution 6", "Receiving Institution 6", "Receiving Entity 6", 5),
    ("Ordering Entity F", "Ordering Institution 2", "Receiving Institution 3", "Receiving Entity 5", 5),
    ("Ordering Entity F", "Ordering Institution 2", "Receiving Institution 1", "Receiving Entity 1", 4),
    ("Ordering Entity F", "Ordering Institution 1", "Receiving Institution 7", "Receiving Entity 7", 3),
]

# Each row now has all eight requested dimensions, plus its ribbon value.
transactions = [
    {
        "ordering_entity": entity_a,
        "ordering_entity_country": ordering_entity_countries[entity_a],
        "ordering_institution": institution_a,
        "ordering_institution_country": ordering_institution_countries[institution_a],
        "receiving_institution": institution_b,
        "receiving_institution_country": receiving_institution_countries[institution_b],
        "receiving_entity": entity_b,
        "receiving_entity_country": receiving_entity_countries[entity_b],
        "value": value,
    }
    for entity_a, institution_a, institution_b, entity_b, value in sample_routes
]

In [29]:
# This insertion order controls the eight left-to-right stages.
column_titles = {
    "ordering_entity": "ORDERING<br>ENTITY",
    "ordering_entity_country": "ORDERING ENTITY<br>COUNTRY",
    "ordering_institution": "ORDERING<br>INSTITUTION",
    "ordering_institution_country": "ORDERING INSTITUTION<br>COUNTRY",
    "receiving_institution": "RECEIVING<br>INSTITUTION",
    "receiving_institution_country": "RECEIVING INSTITUTION<br>COUNTRY",
    "receiving_entity": "RECEIVING<br>ENTITY",
    "receiving_entity_country": "RECEIVING ENTITY<br>COUNTRY",
}

layers = {
    field: list(dict.fromkeys(transaction[field] for transaction in transactions))
    for field in column_titles
}

In [30]:
# Ribbons keep the receiving entity's country colour across all seven hops.
country_colors = {
    "United States": "#ef626c",
    "United Kingdom": "#ff9fb2",
    "Singapore": "#f2c94c",
    "Switzerland": "#59b3b3",
    "Germany": "#a56f4f",
    "Netherlands": "#9b6aa6",
    "France": "#66a866",
}

layer_colors = {
    "ordering_entity": "#4c78a8",
    "ordering_entity_country": "#8297b5",
    "ordering_institution": "#3f7cac",
    "ordering_institution_country": "#8297b5",
    "receiving_institution": "#59b3b3",
    "receiving_institution_country": "#8297b5",
    "receiving_entity": "#d1495b",
    "receiving_entity_country": "#8297b5",
}


def hex_to_rgba(hex_color: str, alpha: float = 0.35) -> str:
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f"rgba({r}, {g}, {b}, {alpha})"

In [31]:
# A wide canvas keeps all eight stages readable; scrolling handles small screens.
chart_width = 1800
x_positions = {
    field: 0.015 + index * (0.97 / (len(layers) - 1))
    for index, field in enumerate(layers)
}

# Match the figure geometry below when estimating Plotly's node heights.
chart_height = 1000
chart_margin = dict(l=30, r=30, t=140, b=40)
node_pad = 24
sankey_y_domain = [0.025, 0.97]
sankey_height = (
    (chart_height - chart_margin["t"] - chart_margin["b"])
    * (sankey_y_domain[1] - sankey_y_domain[0])
)

# Every transaction contributes its value once to each layer's node.
node_totals = defaultdict(float)
for transaction in transactions:
    for layer_name in layers:
        node_totals[(layer_name, transaction[layer_name])] += transaction["value"]

# Sankey uses one common pixels-per-value scale. The most crowded column
# limits it after reserving node_pad between nodes.
value_scale = min(
    (sankey_height - node_pad * (len(labels) - 1))
    / sum(node_totals[(layer_name, label)] for label in labels)
    for layer_name, labels in layers.items()
)

node_ids = []
node_labels = []
node_hover_details = []
node_colors = []
node_x = []
node_y = []

for layer_name, layer_values in layers.items():
    heights = [node_totals[(layer_name, label)] * value_scale for label in layer_values]
    # Distribute spare space between nodes, accounting for their real heights.
    gap = (sankey_height - sum(heights)) / (len(heights) - 1) if len(heights) > 1 else 0
    cursor = 0.0 if len(heights) > 1 else (sankey_height - heights[0]) / 2

    for label, height in zip(layer_values, heights):
        node_ids.append((layer_name, label))
        node_labels.append(label)
        node_hover_details.append(column_titles[layer_name].replace("<br>", " ").title())
        node_x.append(x_positions[layer_name])
        node_y.append((cursor + height / 2) / sankey_height)
        cursor += height + gap

        node_colors.append(layer_colors[layer_name])

node_to_index = {node_id: index for index, node_id in enumerate(node_ids)}

In [32]:
sources = []
targets = []
values = []
link_colors = []
link_hover_details = []

for transaction in transactions:
    path = [(field, transaction[field]) for field in layers]
    link_color = hex_to_rgba(country_colors[transaction["receiving_entity_country"]], alpha=0.32)
    route_detail = "<br>".join(
        f"{column_titles[field].replace('<br>', ' ').title()}: {transaction[field]}"
        for field in layers
    )

    for source_node, target_node in zip(path[:-1], path[1:]):
        sources.append(node_to_index[source_node])
        targets.append(node_to_index[target_node])
        values.append(transaction["value"])
        link_colors.append(link_color)
        link_hover_details.append(route_detail)

In [33]:
fig = go.Figure(
    data=[
        go.Sankey(
            # Keep two-dimensional dragging enabled. Positions are starting points.
            arrangement="freeform",
            domain=dict(y=sankey_y_domain),
            node=dict(
                pad=node_pad,
                thickness=18,
                line=dict(width=0.5, color="rgba(255,255,255,0.35)"),
                label=node_labels,
                customdata=node_hover_details,
                hovertemplate="<b>%{label}</b><br>%{customdata}<br>%{value:,.0f} synthetic transactions<extra></extra>",
                color=node_colors,
                x=node_x,
                y=node_y,
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values,
                color=link_colors,
                customdata=link_hover_details,
                hovertemplate="%{customdata}<br><b>%{value:,.0f} synthetic transactions</b><extra></extra>",
            ),
        )
    ]
)

fig.update_layout(
    title=dict(
        text="Synthetic transaction flow by entity, institution and country",
        x=0.025,
        xanchor="left",
        y=0.975,
        yanchor="top",
    ),
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(size=12, color="#f5f7fa"),
    width=chart_width,
    height=chart_height,
    autosize=True,
    margin=chart_margin,
    annotations=[
        dict(
            x=x_positions[layer_name], y=1.055,
            xref="paper", yref="paper",
            xanchor="left" if layer_name == "ordering_entity" else "right" if layer_name == "receiving_entity_country" else "center",
            yanchor="bottom",
            text=f"<b>{title}</b>",
            showarrow=False,
            font=dict(size=12),
        )
        for layer_name, title in column_titles.items()
    ],
)

fig.show()

In [34]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "20261003_sankey_test.html"

plot_html = fig.to_html(
    include_plotlyjs=False,
    full_html=False,
    config={
        "responsive": True,
        "displaylogo": False,
    },
)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Sankey | phey.app</title>
  <style>
    html,
    body {{
      margin: 0;
      padding: 0;
      min-height: 100%;
      background: #0d1117;
      color: #f5f7fa;
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    }}

    .plot-page {{
      min-height: 100vh;
      background: #0d1117;
      padding: clamp(0.75rem, 2vw, 1.5rem);
      box-sizing: border-box;
    }}

    .plot-wrap {{
      width: 100%;
      margin: 0 auto;
    }}

    .plot-scroll {{
      overflow-x: auto;
    }}

    .plotly-graph-div {{
      min-width: {chart_width}px;
    }}

    .plot-note {{
      color: #aebacd;
      font-size: 0.9rem;
      line-height: 1.5;
    }}

    a.back-link {{
      display: inline-flex;
      margin-bottom: 1rem;
      color: #f5f7fa;
      text-decoration: none;
      opacity: 0.75;
      font-size: 0.95rem;
    }}

    a.back-link:hover {{
      opacity: 1;
    }}
  </style>
</head>
<body>
  <main class="plot-page">
    <div class="plot-wrap">
      <a class="back-link" href="../index.html">← Back to phey.app</a>
      <p class="plot-note">Synthetic transaction counts and country assignments. Ribbon colour follows the receiving entity country. Drag nodes freely; scroll horizontally to see all eight stages.</p>
      <div class="plot-scroll">{plot_html}</div>
    </div>
  </main>
</body>
</html>
"""

# Shared legal navigation and local Plotly dependency survive regeneration.
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from site_export import write_plot_page
write_plot_page(output_file, html)

print(f"Exported Sankey to: {output_file.resolve()}")

Exported Sankey to: C:\Users\PH\Python\p00\output\20261003_sankey_test.html
